#  Verify GPU & CUDA

In [ ]:
import torch

print("CUDA Available     :", torch.cuda.is_available())
print("GPU Name           :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
print("CUDA Version       :", torch.version.cuda)
print("Total VRAM (GB)    :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if torch.cuda.is_available() else "N/A")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device       :", device)

ModuleNotFoundError: No module named 'torch'

# Imports

In [ ]:
import json
import torch
import numpy as np
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast          # FP16 core
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from transformers import BertTokenizer, BertModel, AdamW, get_linear_schedule_with_warmup
from transformers import (
    BertTokenizer,
    BertModel,
    get_linear_schedule_with_warmup,
)
from transformers.optimization import AdamW

# Load & Inspect Data

In [ ]:
with open("supply_chain_disruption_dataset.json") as f:
    raw_data = json.load(f)

print(f"Total samples : {len(raw_data)}")
print("Sample record :", json.dumps(raw_data[0], indent=2))

texts            = [d["text"]           for d in raw_data]
disruption_types = [d["disruption_type"] for d in raw_data]
severities       = [d["severity"]        for d in raw_data]

Total samples : 172
Sample record : {
  "text": "A magnitude 6.8 earthquake in central Japan disrupted semiconductor wafer production at Renesas Electronics' Naka factory, forcing a complete shutdown of clean room operations. The tremor damaged automated wafer handling equipment and triggered emergency safety protocols across the facility. Renesas supplies microcontrollers to several major automakers, and both General Motors and Stellantis confirmed they were evaluating the impact on their chip inventories. The company estimated that full production recovery would take approximately eight weeks.",
  "disruption_type": "natural_disaster",
  "severity": "high",
  "affected_companies": [
    "Renesas Electronics",
    "General Motors",
    "Stellantis"
  ]
}


# Encode Labels

In [ ]:
dt_encoder  = LabelEncoder()
sev_encoder = LabelEncoder()

dt_labels  = dt_encoder.fit_transform(disruption_types)
sev_labels = sev_encoder.fit_transform(severities)

print("Disruption type classes :", list(dt_encoder.classes_))
print("Severity classes        :", list(sev_encoder.classes_))
print("Shapes → dt:", dt_labels.shape, "  sev:", sev_labels.shape)

Disruption type classes : [np.str_('cyber_attack'), np.str_('geopolitical'), np.str_('labor'), np.str_('logistics'), np.str_('natural_disaster'), np.str_('none'), np.str_('operational')]
Severity classes        : [np.str_('high'), np.str_('low'), np.str_('medium'), np.str_('none')]
Shapes → dt: (172,)   sev: (172,)


# Train / Validation Split


In [ ]:
(X_train, X_val,
 dt_train, dt_val,
 sev_train, sev_val) = train_test_split(
    texts, dt_labels, sev_labels,
    test_size=0.2,
    random_state=42,
    stratify=dt_labels,
)

print(f"Train : {len(X_train)}  |  Val : {len(X_val)}")

Train : 137  |  Val : 35


# Tokenizer

In [ ]:
MODEL_NAME = "bert-base-uncased"
tokenizer  = BertTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts_list, max_len=256):
    return tokenizer(
        texts_list,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )

train_enc = tokenize(X_train)
val_enc   = tokenize(X_val)

print("Train input_ids shape :", train_enc["input_ids"].shape)
print("Val   input_ids shape :", val_enc["input_ids"].shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train input_ids shape : torch.Size([137, 256])
Val   input_ids shape : torch.Size([35, 256])


# PyTorch Dataset & DataLoader


In [ ]:
class SupplyChainDataset(Dataset):
    def __init__(self, encodings, dt_labels, sev_labels):
        self.encodings  = encodings
        self.dt_labels  = torch.tensor(dt_labels,  dtype=torch.long)
        self.sev_labels = torch.tensor(sev_labels, dtype=torch.long)

    def __len__(self):
        return len(self.dt_labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["dt_label"]  = self.dt_labels[idx]
        item["sev_label"] = self.sev_labels[idx]
        return item


BATCH_SIZE = 32                       # increase if VRAM allows (e.g. 32)

train_dataset = SupplyChainDataset(train_enc, dt_train, sev_train)
val_dataset   = SupplyChainDataset(val_enc,   dt_val,   sev_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  pin_memory=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          pin_memory=True, num_workers=2)

# pin_memory=True + num_workers speeds up GPU data transfers

#  Multi-Task BERT Model (FP16-Ready)

In [ ]:
class BertMultiTask(nn.Module):
    """
    Shared BERT backbone → two classification heads:
      • head 1 : disruption_type  (6 classes)
      • head 2 : severity         (3 classes)
    """
    def __init__(self, model_name, num_dt, num_sev, dropout=0.3):
        super().__init__()
        self.bert    = BertModel.from_pretrained(model_name)
        hidden       = self.bert.config.hidden_size          # 768

        self.dropout  = nn.Dropout(dropout)
        self.dt_head  = nn.Linear(hidden, num_dt)
        self.sev_head = nn.Linear(hidden, num_sev)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out    = self.bert(input_ids=input_ids,
                           attention_mask=attention_mask,
                           token_type_ids=token_type_ids)
        pooled = self.dropout(out.pooler_output)             # [B, 768]
        return self.dt_head(pooled), self.sev_head(pooled)


NUM_DT  = len(dt_encoder.classes_)   # 6
NUM_SEV = len(sev_encoder.classes_)  # 3

model = BertMultiTask(MODEL_NAME, NUM_DT, NUM_SEV).to(device)
print(model)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertMultiTask(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_aff

# Optimizer, Scheduler & FP16 Scaler

In [ ]:
EPOCHS = 50
LR     = 2e-5

# Separate LR for backbone vs heads (optional but recommended)
optimizer = AdamW([
    {"params": model.bert.parameters(),    "lr": LR},
    {"params": model.dt_head.parameters(), "lr": LR * 5},
    {"params": model.sev_head.parameters(),"lr": LR * 5},
], weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

criterion = nn.CrossEntropyLoss()

# ── FP16 GradScaler — prevents underflow in half-precision gradients ──
scaler = GradScaler()

print(f"Total training steps : {total_steps}")
print(f"Warmup steps         : {int(0.1 * total_steps)}")

Total training steps : 450
Warmup steps         : 45


/tmp/ipykernel_7896/1641098900.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


#  Training Loop with FP16

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, scaler, criterion):
    model.train()
    total_loss, correct_dt, correct_sev, total = 0, 0, 0, 0

    for batch in loader:
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        dt_true        = batch["dt_label"].to(device, non_blocking=True)
        sev_true       = batch["sev_label"].to(device, non_blocking=True)

        optimizer.zero_grad()

        # ── Forward pass in FP16 ──────────────────────────────────────
        with autocast():
            dt_logits, sev_logits = model(input_ids, attention_mask)
            loss = criterion(dt_logits, dt_true) + \
                   criterion(sev_logits, sev_true)

        # ── Backward pass with loss scaling ──────────────────────────
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss  += loss.item()
        correct_dt  += (dt_logits.argmax(1)  == dt_true).sum().item()
        correct_sev += (sev_logits.argmax(1) == sev_true).sum().item()
        total       += dt_true.size(0)

    avg_loss   = total_loss / len(loader)
    acc_dt     = correct_dt  / total * 100
    acc_sev    = correct_sev / total * 100
    return avg_loss, acc_dt, acc_sev

# Validation Loop


In [ ]:
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_dt_pred, all_dt_true   = [], []
    all_sev_pred, all_sev_true = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            dt_true        = batch["dt_label"].to(device, non_blocking=True)
            sev_true       = batch["sev_label"].to(device, non_blocking=True)

            with autocast():                                   # FP16 here too
                dt_logits, sev_logits = model(input_ids, attention_mask)
                loss = criterion(dt_logits, dt_true) + \
                       criterion(sev_logits, sev_true)

            total_loss += loss.item()
            all_dt_pred.extend(dt_logits.argmax(1).cpu().numpy())
            all_dt_true.extend(dt_true.cpu().numpy())
            all_sev_pred.extend(sev_logits.argmax(1).cpu().numpy())
            all_sev_true.extend(sev_true.cpu().numpy())

    avg_loss = total_loss / len(loader)
    return avg_loss, all_dt_pred, all_dt_true, all_sev_pred, all_sev_true

# Run Training

In [ ]:
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc_dt, tr_acc_sev = train_epoch(
        model, train_loader, optimizer, scheduler, scaler, criterion
    )
    val_loss, dt_pred, dt_true, sev_pred, sev_true = validate(
        model, val_loader, criterion
    )

    print(f"\nEpoch {epoch}/{EPOCHS}")
    print(f"  Train → Loss: {tr_loss:.4f}  |  DT Acc: {tr_acc_dt:.1f}%  |  Sev Acc: {tr_acc_sev:.1f}%")
    print(f"  Val   → Loss: {val_loss:.4f}")

    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "bert_supply_chain_best.pt")
        print("  ✅ Best model saved!")

    # GPU memory usage
    print(f"  GPU Mem Used : {torch.cuda.memory_allocated() / 1e9:.2f} GB  "
          f"| Reserved : {torch.cuda.memory_reserved() / 1e9:.2f} GB")

/tmp/ipykernel_7896/3577844918.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



Epoch 1/50
  Train → Loss: 3.3688  |  DT Acc: 16.1%  |  Sev Acc: 33.6%
  Val   → Loss: 3.2643
  ✅ Best model saved!
  GPU Mem Used : 1.81 GB  | Reserved : 3.41 GB

Epoch 2/50
  Train → Loss: 3.2551  |  DT Acc: 16.1%  |  Sev Acc: 41.6%
  Val   → Loss: 3.1753
  ✅ Best model saved!
  GPU Mem Used : 1.81 GB  | Reserved : 3.41 GB

Epoch 3/50
  Train → Loss: 3.1082  |  DT Acc: 25.5%  |  Sev Acc: 45.3%
  Val   → Loss: 3.0438
  ✅ Best model saved!
  GPU Mem Used : 1.81 GB  | Reserved : 3.41 GB

Epoch 4/50
  Train → Loss: 2.9487  |  DT Acc: 35.8%  |  Sev Acc: 43.8%
  Val   → Loss: 2.9863
  ✅ Best model saved!
  GPU Mem Used : 1.81 GB  | Reserved : 3.41 GB

Epoch 5/50
  Train → Loss: 2.6464  |  DT Acc: 42.3%  |  Sev Acc: 53.3%
  Val   → Loss: 2.6089
  ✅ Best model saved!
  GPU Mem Used : 1.81 GB  | Reserved : 3.41 GB

Epoch 6/50
  Train → Loss: 2.4355  |  DT Acc: 56.9%  |  Sev Acc: 54.0%
  Val   → Loss: 2.3695
  ✅ Best model saved!
  GPU Mem Used : 1.81 GB  | Reserved : 3.41 GB

Epoch 7/50
  Tr

# Full Evaluation Report

In [ ]:
# Load best model before evaluating
model.load_state_dict(torch.load("bert_supply_chain_best.pt"))

_, dt_pred, dt_true, sev_pred, sev_true = validate(model, val_loader, criterion)

print("=" * 55)
print("DISRUPTION TYPE — Classification Report")
print("=" * 55)
print(classification_report(dt_true, dt_pred,
                             target_names=dt_encoder.classes_))

print("=" * 55)
print("SEVERITY — Classification Report")
print("=" * 55)
print(classification_report(sev_true, sev_pred,
                             target_names=sev_encoder.classes_))

/tmp/ipykernel_7896/3594523145.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():                                   # FP16 here too


DISRUPTION TYPE — Classification Report
                  precision    recall  f1-score   support

    cyber_attack       1.00      1.00      1.00         5
    geopolitical       1.00      1.00      1.00         6
           labor       1.00      0.83      0.91         6
       logistics       0.80      1.00      0.89         4
natural_disaster       1.00      1.00      1.00         6
            none       1.00      1.00      1.00         2
     operational       1.00      1.00      1.00         6

        accuracy                           0.97        35
       macro avg       0.97      0.98      0.97        35
    weighted avg       0.98      0.97      0.97        35

SEVERITY — Classification Report
              precision    recall  f1-score   support

        high       0.60      0.25      0.35        12
         low       0.50      0.33      0.40         3
      medium       0.62      0.89      0.73        18
        none       1.00      1.00      1.00         2

    accuracy  

# Save Model & Tokenizer


In [ ]:
# Save full model state + encoders
torch.save({
    "model_state_dict" : model.state_dict(),
    "dt_classes"       : list(dt_encoder.classes_),
    "sev_classes"      : list(sev_encoder.classes_),
}, "bert_supply_chain_final.pt")

tokenizer.save_pretrained("bert_supply_chain_tokenizer/")
print("Model + tokenizer saved.")

Model + tokenizer saved.


# Inference on New Text

In [ ]:
def predict(text: str) -> dict:
    model.eval()
    enc = tokenize([text])
    with torch.no_grad():
        with autocast():                                       # FP16 inference
            dt_logits, sev_logits = model(
                enc["input_ids"].to(device),
                enc["attention_mask"].to(device),
            )
    dt_prob  = torch.softmax(dt_logits,  dim=1).cpu().numpy()[0]
    sev_prob = torch.softmax(sev_logits, dim=1).cpu().numpy()[0]

    dt_pred  = dt_encoder.classes_[dt_logits.argmax().item()]
    sev_pred = sev_encoder.classes_[sev_logits.argmax().item()]

    return {
        "disruption_type"      : dt_pred,
        "disruption_confidence": f"{dt_prob.max()*100:.1f}%",
        "severity"             : sev_pred,
        "severity_confidence"  : f"{sev_prob.max()*100:.1f}%",
    }


# Test it
sample = "A cyberattack on a major logistics firm encrypted shipment data across Europe."
print(predict(sample))
# e.g. {'disruption_type': 'cyber_attack', 'disruption_confidence': '94.2%',
#        'severity': 'high',  'severity_confidence': '88.7%'}

{'disruption_type': np.str_('cyber_attack'), 'disruption_confidence': '83.9%', 'severity': np.str_('high'), 'severity_confidence': '88.4%'}


/tmp/ipykernel_7896/3146017837.py:5: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():                                       # FP16 inference
